# Set up colab gpu runtime env

In [ ]:
!pip install segmentation-models-pytorch
!pip install -U git+https://github.com/albumentations-team/albumentations
!pip install --upgrade opencv-contrib-python

#Utility function

In [ ]:
class Utils:
    @staticmethod
    def display_image(image, true_mask, pred_mask=None):
        # Handle tensor inputs: move to CPU, convert to numpy, permute dimensions
        if isinstance(image, torch.Tensor):
            image = image.permute(1, 2, 0).cpu().numpy()
        if isinstance(true_mask, torch.Tensor):
            true_mask = true_mask.squeeze().cpu().numpy() # Squeeze to (H,W) if (1,H,W)

        f, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

        ax1.set_title('image')
        ax1.imshow(image)

        ax2.set_title('Groud truth')
        ax2.imshow(true_mask, cmap='gray')

        plt.show()

    @staticmethod
    def display_inference(image, true_mask, pred_mask):
        # Ensure inputs are tensors and handle their display format
        if isinstance(image, torch.Tensor):
            image = image.permute(1, 2, 0).cpu().numpy()
        if isinstance(true_mask, torch.Tensor):
            true_mask = true_mask.squeeze().cpu().numpy()
        if isinstance(pred_mask, torch.Tensor):
            pred_mask = pred_mask.squeeze().cpu().numpy()

        f, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5))

        ax1.set_title('Original Image')
        ax1.imshow(image)

        ax2.set_title('Ground Truth')
        ax2.imshow(true_mask, cmap='gray')

        ax3.set_title('Predicted Mask')
        ax3.imshow(pred_mask, cmap='gray')

        plt.show()

# Download Dataset

In [ ]:
!git clone https://github.com/parth1620/Road_seg_dataset.git

In [ ]:
import os
import sys

In [ ]:
import torch
import cv2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

import helper

#Setup Config

In [ ]:
CSV_PATH = 'Road_seg_dataset/train.csv'
DATA_DIR = 'Road_seg_dataset'

EPOCHS = 30
LR = 0.001
BATCH_SIZE = 8
IMG_SIZE = 512

ENCODER = 'resnet34'
WEIGHTS =  'imagenet'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# load dataset with pandas
df = pd.read_csv(CSV_PATH)
df.head()

In [ ]:
# some visualizations

idx = 3
row = df.iloc[idx]

image_path = os.path.join(DATA_DIR, row.images)
mask_path = os.path.join(DATA_DIR, row.masks)

image = cv2.imread(image_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE) /255

#we divide mask/255 to change the scale from 0 to 255 to 0 to 1


In [ ]:
f, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

ax1.set_title('image')
ax1.imshow(image)

ax2.set_title('Groud truth')
ax2.imshow(mask)

plt.show()

In [ ]:
## split the data for training and validation

train_df, valid_df = train_test_split(df, test_size=0.2, random_state=42)
print(f'train size : {len(train_df)}, valid size : {len(valid_df)}')


#Augmentation Functions

In [ ]:
import albumentations as A
def get_train_augs():
  return A.Compose([
      A.Resize(IMG_SIZE, IMG_SIZE),
      A.HorizontalFlip(p=0.5),
      A.VerticalFlip(p=0.5)
  ])

def get_valid_augs():
  return A.Compose([
      A.Resize(IMG_SIZE, IMG_SIZE)
  ])

#Create Custom Dataset

In [ ]:
from torch.utils.data import Dataset

class SegmentationDataset(Dataset):
  def __init__(self, df, augmentations):
    self.df = df
    self.augmentations = augmentations

  def __len__(self):
    return len(self.df)

  def __getitem__(self, idx):
    row = self.df.iloc[idx]

    image_path = os.path.join(DATA_DIR, row.images)
    mask_path = os.path.join(DATA_DIR, row.masks)
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    mask = np.expand_dims(mask, axis=-1) # (h,w,c)

    if self.augmentations:
      data = self.augmentations(image=image, mask=mask)
      image = data['image']
      mask = data['mask']

    image = np.transpose(image, (2, 0, 1)).astype(np.float32)  #(c,h,w)
    mask = np.transpose(mask, (2, 0, 1)).astype(np.float32) #(c,h,w)
    # Convert image and masl to torch tensor
    image = torch.tensor(image, dtype=torch.float) / 255.0
    mask = torch.tensor(mask, dtype=torch.float) / 255.0
    return image, mask


In [ ]:
trainset = SegmentationDataset(train_df, get_train_augs())
validset = SegmentationDataset(valid_df, get_valid_augs())

In [ ]:
#plot an example for visualization
idx = 15
image, mask =  trainset[idx]
Utils.display_image(image, mask)

#Load dataset into batches


In [ ]:
trainloader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True)
validloader = DataLoader(validset, batch_size=BATCH_SIZE)

In [ ]:
#print size of the first tensor in trainset

for image, mask in trainloader:
  print(image.shape) # format -> (batch,c,h,w)
  print(mask.shape) # format -> (batch,c,h,w)
  break

# Create NN  (Segmentation model)

In [ ]:
#load UNet architecture
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.losses import DiceLoss

from torch import nn

In [ ]:
class SegmentationModel(nn.Module):
  def __init__(self):
    super(SegmentationModel, self).__init__()
    self.backbone = smp.Unet(
        encoder_name=ENCODER,
        encoder_weights=WEIGHTS,
        in_channels=3,
        classes=1,
        activation=None
    )
  def forward(self, images, masks=None):
      logits = self.backbone(images)
      if masks != None:
        loss1 = DiceLoss(mode='binary')(logits, masks)
        loss2 = nn.BCEWithLogitsLoss()(logits, masks)
        return logits, loss1 + loss2

      return logits

In [ ]:
model = SegmentationModel()
model.to(DEVICE);

In [ ]:
# Instantiate the segmentation model.
model = SegmentationModel()
model.to(DEVICE);

#Create Train and Validation Functions

In [ ]:

def train_fc(dataloader, model, optimizer):
  model.train()
  total_loss = 0.0
  for images, masks in dataloader:
    images = images.to(DEVICE)
    masks = masks.to(DEVICE)

    optimizer.zero_grad()

    logits, loss = model(images, masks)
    loss.backward()
    optimizer.step()

    total_loss += loss.item()

  return total_loss / len(dataloader)

def valid_fc(dataloader, model):
  model.eval()
  total_loss = 0.0
  with torch.no_grad():
    for images, masks in dataloader:
      images = images.to(DEVICE)
      masks = masks.to(DEVICE)
      logits, loss = model(images, masks)
      total_loss += loss.item()
      return total_loss / len(dataloader)


#Train Model

In [ ]:
best_loss = np.inf

optimizer = torch.optim.Adam(model.parameters(), lr=LR)

for epoch in range(EPOCHS):
  train_loss = train_fc(trainloader, model, optimizer)
  valid_loss = valid_fc(validloader, model)
  if valid_loss < best_loss:
    torch.save(model.state_dict(), 'best_model.pth')
    print(f'Saved best model with validation loss {valid_loss}')
    best_loss = valid_loss

  print(f'Epoch : {epoch+1} Train Loss : {train_loss:.4f} Valid_Loss : {valid_loss:.4f}')

#Inference

In [ ]:
idx = 30
model.load_state_dict(torch.load('best_model.pth'))
image, mask = validset[idx]

logit_mask = model(image.to(DEVICE).unsqueeze(0)) # c,h,w, -> b,c,h,w
pred_mask = torch.sigmoid(logit_mask)
pred_mask = (pred_mask > 0.5) * 1.0

In [ ]:
Utils.display_inference(image, mask, pred_mask)

#Extra documents for help


albumentation documentation : https://albumentations.ai/docs/


segmentation_models_pytorch documentation : https://smp.readthedocs.io/en/latest/

Created by Carlos Sarasty.